## DICOM --> PNG CONVERSION

In [1]:
from pathlib import Path
import pydicom


BASE_DIR = Path(".").resolve()
DATASET = BASE_DIR / "Datasets"
WORKSPACE_DIR = BASE_DIR.parent

In [2]:
DATASET_ROOTS = [
    DATASET / "Aliza",
    DATASET / "AIIMS",
    DATASET / "Breast_Lesions_USG",
    DATASET / "MONAI",
    DATASET / "Heartcycle",
    DATASET / "UTA7",
    DATASET / "UTA10",
    DATASET / "UTA4",
    DATASET / "ReMIND",
    DATASET / "TCIA_IDC_Prostate_MRI_US_Biopsy",
    
]


# ------------------------------------------------------------
# OUTPUT CSV
# ------------------------------------------------------------

OUT_CSV = BASE_DIR / "interoperability_evaluation_us_dicom.csv"


# ============================================================
# STRICT DICOM DETECTION
# ============================================================

def is_dicom_file(path: Path) -> bool:
    """
    Strict DICOM validation.

    Prevents false positives from:
    - PNG
    - JPG
    - MP4
    - random binary files
    """

    try:

        # Standard DICOM preamble check
        with open(path, "rb") as f:
            f.seek(128)
            magic = f.read(4)

        if magic == b"DICM":
            return True

        # Secondary strict validation
        ds = pydicom.dcmread(
            str(path),
            stop_before_pixels=True,
            force=False
        )

        # Essential DICOM tags must exist
        required_tags = [
            "PatientID",
            "StudyInstanceUID",
            "SeriesInstanceUID",
            "SOPInstanceUID",
        ]

        return any(hasattr(ds, tag) for tag in required_tags)

    except Exception:
        return False


# ============================================================
# COLLECT DICOM FILES
# ============================================================

def collect_dicom_files(roots):

    found = []

    print("Collecting DICOM files...\n")

    for root in roots:

        if not root.exists():
            print(f"[NOT FOUND] {root}")
            continue

        print(f"Scanning: {root}")

        all_files = [f for f in root.rglob("*") if f.is_file()]

        before_count = len(found)

        for f in all_files:

            # Common DICOM extensions
            if f.suffix.lower() in [".dcm", ".dicom"]:
                if is_dicom_file(f):
                    found.append(f)
                continue

            # Extensionless anonymized DICOMs
            if f.suffix == "":
                if is_dicom_file(f):
                    found.append(f)

        added = len(found) - before_count

        print(f"  Found {added} DICOM files\n")

    return sorted(found)


# ============================================================
# RUN COLLECTION
# ============================================================

candidate_files = collect_dicom_files(DATASET_ROOTS)

print("=" * 60)
print(f"TOTAL DICOM FILES FOUND: {len(candidate_files)}")
print("=" * 60)


Scanning: C:\Users\Gajendra\Desktop\D3_MONAI\Datasets\Aliza
  Found 22 DICOM files

Scanning: C:\Users\Gajendra\Desktop\D3_MONAI\Datasets\AIIMS
  Found 7 DICOM files

Scanning: C:\Users\Gajendra\Desktop\D3_MONAI\Datasets\Breast_Lesions_USG
  Found 0 DICOM files

Scanning: C:\Users\Gajendra\Desktop\D3_MONAI\Datasets\MONAI
  Found 0 DICOM files

Scanning: C:\Users\Gajendra\Desktop\D3_MONAI\Datasets\Heartcycle
  Found 0 DICOM files

Scanning: C:\Users\Gajendra\Desktop\D3_MONAI\Datasets\UTA7
  Found 198 DICOM files

Scanning: C:\Users\Gajendra\Desktop\D3_MONAI\Datasets\UTA10
  Found 27 DICOM files

Scanning: C:\Users\Gajendra\Desktop\D3_MONAI\Datasets\UTA4
  Found 1132 DICOM files

Scanning: C:\Users\Gajendra\Desktop\D3_MONAI\Datasets\ReMIND
  Found 320 DICOM files

Scanning: C:\Users\Gajendra\Desktop\D3_MONAI\Datasets\TCIA_IDC_Prostate_MRI_US_Biopsy
  Found 1762 DICOM files

TOTAL DICOM FILES FOUND: 3468


In [3]:
# ============================================================
# DICOM --> PNG CONVERSION
# Preserves:
#   - Folder hierarchy
#   - Multi-frame structure
#   - DICOM display rendering
#   - Original bit depth whenever possible
# ============================================================

import numpy as np
from pathlib import Path
from PIL import Image

import pydicom
from pydicom.pixel_data_handlers.util import (
    apply_voi_lut,
    convert_color_space
)

# ============================================================
# OUTPUT DIRECTORY
# ============================================================
BASE_DIR = Path(".").resolve()
DATASET = BASE_DIR / "Datasets"
OUTPUT_ROOT = BASE_DIR / "post_processed_data"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# ============================================================
# SAVE FRAME FUNCTION
# ============================================================

def save_frame(frame, save_path):
    """
    Save frame preserving dtype whenever possible.
    """

    save_path.parent.mkdir(parents=True, exist_ok=True)

    # --------------------------------------------------------
    # FLOAT TYPES
    # PNG does not support float directly
    # --------------------------------------------------------

    if np.issubdtype(frame.dtype, np.floating):

        frame = frame.astype(np.float32)

        frame_min = frame.min()
        frame_max = frame.max()

        if frame_max > frame_min:
            frame = (frame - frame_min) / (frame_max - frame_min)

        frame = (frame * 65535).astype(np.uint16)

    # --------------------------------------------------------
    # SIGNED INTEGER
    # --------------------------------------------------------

    elif np.issubdtype(frame.dtype, np.signedinteger):

        frame = frame.astype(np.int32)

        frame = frame - frame.min()

        if frame.max() <= 255:
            frame = frame.astype(np.uint8)
        else:
            frame = frame.astype(np.uint16)

    # --------------------------------------------------------
    # UNSIGNED INTEGER
    # --------------------------------------------------------

    elif np.issubdtype(frame.dtype, np.unsignedinteger):

        if frame.dtype not in [np.uint8, np.uint16]:

            if frame.max() <= 255:
                frame = frame.astype(np.uint8)
            else:
                frame = frame.astype(np.uint16)

    # --------------------------------------------------------
    # FINAL SAVE
    # --------------------------------------------------------

    img = Image.fromarray(frame)
    img.save(save_path)

# ============================================================
# PROCESS DICOM
# ============================================================

print("\nStarting DICOM --> PNG conversion...\n")

for dcm_file in candidate_files:

    try:
        # ----------------------------------------------------
        # READ DICOM
        # ----------------------------------------------------
        ds = pydicom.dcmread(str(dcm_file), force=True)
        arr = ds.pixel_array
        # ----------------------------------------------------
        # APPLY DISPLAY TRANSFORMATIONS
        # Similar to DICOM viewers
        # ----------------------------------------------------

        try:
            arr = apply_voi_lut(arr, ds)
        except:
            pass
        photometric = ds.get("PhotometricInterpretation", "")
        # MONOCHROME1 needs inversion
        if photometric == "MONOCHROME1":
            arr = np.max(arr) - arr

        # Convert YBR to RGB
        elif photometric.startswith("YBR"):
            try:
                arr = convert_color_space(arr, photometric, "RGB")
            except:
                pass

        # ----------------------------------------------------
        # PRESERVE HIERARCHY
        # ----------------------------------------------------

        try:
            relative_path = dcm_file.relative_to(DATASET)
        except:
            relative_path = Path(dcm_file.name)
        out_dir = OUTPUT_ROOT / relative_path.parent
        base_name = dcm_file.stem
        # ----------------------------------------------------
        # SINGLE FRAME
        # ----------------------------------------------------

        is_single_frame = (
            arr.ndim == 2 or
            (
                arr.ndim == 3 and
                arr.shape[-1] in [3, 4]
            )
        )
        if is_single_frame:
            png_path = out_dir / f"{base_name}.png"
            save_frame(arr, png_path)
        # ----------------------------------------------------
        # MULTI FRAME
        # ----------------------------------------------------

        else:

            multi_dir = out_dir / base_name
            multi_dir.mkdir(parents=True, exist_ok=True)

            for i, frame in enumerate(arr):

                png_path = multi_dir / f"{base_name}_{i:04d}.png"

                save_frame(frame, png_path)

        print(f"[OK] {dcm_file}")

    except Exception as e:

        print(f"[FAILED] {dcm_file}")
        print(f"         {e}")

print("\nPNG conversion completed.\n")


Starting DICOM --> PNG conversion...

[OK] C:\Users\Gajendra\Desktop\D3_MONAI\Datasets\AIIMS\AIIMSJ_GE_Vivid_E95\Q3EDC380.dcm
[OK] C:\Users\Gajendra\Desktop\D3_MONAI\Datasets\AIIMS\AIIMSJ_GE_Vivid_E95\Q3EDCA02.dcm
[OK] C:\Users\Gajendra\Desktop\D3_MONAI\Datasets\AIIMS\AIIMSJ_GE_Vivid_E95\Q3EDCBG4.dcm
[OK] C:\Users\Gajendra\Desktop\D3_MONAI\Datasets\AIIMS\AIIMSJ_GE_Vivid_E95\Q3EDCH86.dcm
[OK] C:\Users\Gajendra\Desktop\D3_MONAI\Datasets\AIIMS\AIIMSJ_GE_Vivid_E95\Q3EDCK08.dcm
[OK] C:\Users\Gajendra\Desktop\D3_MONAI\Datasets\AIIMS\AIIMSJ_Philips_EPIQ_7C\IM_0001.dcm
[OK] C:\Users\Gajendra\Desktop\D3_MONAI\Datasets\AIIMS\AIIMSJ_Philips_EPIQ_7C\IM_0002.dcm
[OK] C:\Users\Gajendra\Desktop\D3_MONAI\Datasets\Aliza\Enhanced_US_Volume_Storage\usvolume_4D.dcm
[OK] C:\Users\Gajendra\Desktop\D3_MONAI\Datasets\Aliza\Ultrasound_Image_Storage\US-GE-4AICL142.dcm
[OK] C:\Users\Gajendra\Desktop\D3_MONAI\Datasets\Aliza\Ultrasound_Image_Storage\US.1.3.46.670589.14.3000.100.2.199999.20110826084400.0.dcm
[OK] 

C:\Users\Gajendra\Desktop\D3_MONAI\monai_d3\lib\site-packages\pydicom\pixel_data_handlers\rle_handler.py:346: UserWarning: The decoded RLE segment contains non-conformant padding - 276025 vs. 276024 bytes expected
  warnings.warn(


[OK] C:\Users\Gajendra\Desktop\D3_MONAI\Datasets\Aliza\Ultrasound_Multiframe_Image_Storage\us_rle_ybr.dcm
[OK] C:\Users\Gajendra\Desktop\D3_MONAI\Datasets\ReMIND\remind\ReMIND-001\90478\23771\8c780494-786d-4ed7-b2a0-24fe18bd8dce.dcm
[OK] C:\Users\Gajendra\Desktop\D3_MONAI\Datasets\ReMIND\remind\ReMIND-001\90478\64615\bf1b0a44-60ca-4d20-a6e2-dde3ae34ea53.dcm
[OK] C:\Users\Gajendra\Desktop\D3_MONAI\Datasets\ReMIND\remind\ReMIND-001\90478\88762\91d91920-3d47-4a81-8a45-3fa9fdb2b70a.dcm
[OK] C:\Users\Gajendra\Desktop\D3_MONAI\Datasets\ReMIND\remind\ReMIND-002\38609\28174\99d9bce9-a4d5-4c53-bdb3-65cbd64b0d80.dcm
[OK] C:\Users\Gajendra\Desktop\D3_MONAI\Datasets\ReMIND\remind\ReMIND-002\38609\30846\0d3220cf-c1e2-45dd-b3e2-af125beb8a15.dcm
[OK] C:\Users\Gajendra\Desktop\D3_MONAI\Datasets\ReMIND\remind\ReMIND-002\38609\95826\39ea0987-7be5-4b9c-aa5c-a538ab0f085a.dcm
[OK] C:\Users\Gajendra\Desktop\D3_MONAI\Datasets\ReMIND\remind\ReMIND-003\75401\29910\495f767d-377c-41ed-8df7-c79c44b23d02.dcm
[OK] 